# Low-Resolution RNA Structure Analysis Pipeline

This notebook demonstrates the clean, refactored low-resolution RNA analysis pipeline.
It reproduces the functionality of the original `low_res_pipeline.ipynb` using the new organized codebase.

## Setup and Imports

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add the parent directory to path so we can import mintage_low_res package
sys.path.append('../..')

# Import the new organized modules
from mintage_low_res.parsing import parse_pdb_files
from mintage_low_res.preprocessing import (
    scale_coordinates, 
    determine_pucker_data,
    spherical_to_vec,
    exponential_map
)
from mintage_low_res.clustering import (
    pre_clustering, 
    refine_clusters_with_pns,
    PNSClusterer
)
from mintage_low_res.analysis import (
    run_low_res_experiments,
    run_pucker_analysis
)
from mintage_low_res.visualization import (
    create_scatter_plots,
    plot_experiment_summary
)
from mintage_low_res.config import ExperimentConfig, PUCKER_TYPES

# Set up plotting
plt.style.use('default')
%matplotlib inline

## Configuration

In [2]:
# Configuration
PDB_DIRECTORY = "/Users/kaisardauletbek/Documents/GitHub/RNA-Classification/data/rna2020_pruned_pdbs/"
OUTPUT_DIR = "results"
CACHE_DIR = "cache"

# Create experiment configuration
config = ExperimentConfig(
    min_cluster_size=3,
    pns_scale=12000,
    pucker_types=['c2c2', 'c2c3', 'c3c2', 'c3c3']
)

print(f"Configuration:")
print(f"  PDB Directory: {PDB_DIRECTORY}")
print(f"  Min Cluster Size: {config.min_cluster_size}")
print(f"  PNS Scale: {config.pns_scale}")
print(f"  Pucker Types: {config.pucker_types}")

Configuration:
  PDB Directory: /Users/kaisardauletbek/Documents/GitHub/RNA-Classification/data/rna2020_pruned_pdbs/
  Min Cluster Size: 3
  PNS Scale: 12000
  Pucker Types: ['c2c2', 'c2c3', 'c3c2', 'c3c3']


## Method 1: Complete Pipeline (Recommended)

Run the complete analysis pipeline with a single function call.

In [3]:
# Run the complete pipeline
results = run_low_res_experiments(
    pdb_directory=PDB_DIRECTORY,
    pucker_types=config.pucker_types,
    config=config,
    cache_dir=CACHE_DIR,
    output_dir=OUTPUT_DIR
)

print("\n=== Pipeline Results ===")
print(f"Total suites parsed: {len(results['suites'])}")
print(f"Scaling factors: λ_d={results['scaling_factors']['lambda_d']:.3f}, λ_α={results['scaling_factors']['lambda_alpha']:.3f}")
print(f"Total execution time: {results['total_time']:.1f} seconds")

# Display results for each pucker type
for pucker_type, pucker_result in results['pucker_results'].items():
    meta = pucker_result['metadata']
    print(f"\n{pucker_type.upper()}:")
    print(f"  Suites: {meta['n_suites']}")
    print(f"  Final clusters: {meta['n_mode_clusters']}")
    print(f"  Analysis time: {meta['analysis_time']:.2f}s")

=== Low-Resolution RNA Analysis Pipeline ===
PDB Directory: /Users/kaisardauletbek/Documents/GitHub/RNA-Classification/data/rna2020_pruned_pdbs/
Pucker Types: ['c2c2', 'c2c3', 'c3c2', 'c3c3']
Min Cluster Size: 3
PNS Scale: 12000

1. Parsing PDB files...
suites_/Users/kaisardauletbek/Documents/GitHub/RNA-Classification/data/rna2020_pruned_pdbs
   Parsed 0 suites

2. Scaling coordinates...


ValueError: No Suite objects provided for coordinate scaling

## Visualize Results

In [ ]:
# Create comprehensive visualization
plot_experiment_summary(results, output_dir=f"{OUTPUT_DIR}/plots")

# Display some plots inline
from IPython.display import Image, display
import os

plot_files = [
    "pucker_statistics.png",
    "timing_summary.png",
    "pucker_cluster_results.png"
]

for plot_file in plot_files:
    plot_path = f"{OUTPUT_DIR}/plots/{plot_file}"
    if os.path.exists(plot_path):
        print(f"\n{plot_file.replace('_', ' ').title()}")
        display(Image(plot_path))

## Method 2: Step-by-Step Analysis (For Understanding)

Run the analysis step by step to understand each component.

In [ ]:
# Step 1: Parse PDB files
print("1. Parsing PDB files...")
suites = parse_pdb_files(PDB_DIRECTORY, cache_dir=CACHE_DIR)
print(f"   Parsed {len(suites)} suites")

# Step 2: Scale coordinates
print("\n2. Scaling coordinates...")
scaled_coords, lambda_d, lambda_alpha = scale_coordinates(
    suites,
    scale_distance_variance=True,
    scale_alpha_variance=False,
    preserve_distance_mean=True,
    preserve_alpha_mean=True
)

print(f"   Scaling factors: λ_d={lambda_d:.3f}, λ_α={lambda_alpha:.3f}")
print(f"   Scaled coordinates shape: {scaled_coords.shape}")

# Step 3: Analyze pucker types
print("\n3. Analyzing pucker types...")
pucker_indices = {}
for pucker in config.pucker_types:
    indices, _ = determine_pucker_data(suites, pucker)
    pucker_indices[pucker] = np.asarray(indices, dtype=int)
    print(f"   {pucker}: {len(indices)} suites")

### Detailed Analysis for One Pucker Type

In [ ]:
# Analyze c3c3 in detail (largest group)
pucker_type = 'c3c3'
print(f"Detailed analysis for {pucker_type.upper()}")

# Get pucker-specific data
indices = pucker_indices[pucker_type]
scaled_coords_subset = scaled_coords[indices]

print(f"Working with {len(scaled_coords_subset)} suites")

# Hierarchical clustering
from scipy.cluster.hierarchy import single as single_linkage

optimal_q_fold = config.get_optimal_q_fold(pucker_type)
print(f"Using optimal q_fold = {optimal_q_fold}")

clusters, outliers, distance_threshold = pre_clustering(
    input_data=scaled_coords_subset,
    m=config.min_cluster_size,
    percentage=0.0,
    string_folder=f"{OUTPUT_DIR}/clustering",
    method=single_linkage,
    q_fold=optimal_q_fold,
    distance="low_res_suite_shape"
)

cluster_sizes = [len(c) for c in clusters]
print(f"Hierarchical clustering: {len(clusters)} clusters with sizes {cluster_sizes}")

### PNS Transformation and Refinement

In [ ]:
# PNS transformation (following the notebook approach)
d2_s, d3_s, alpha_s, theta1, phi1, theta2, phi2 = scaled_coords_subset.T

print("Applying PNS transformations...")

# Transform spherical coordinates
pns_clusterer = PNSClusterer(scale=config.pns_scale)

# Transform first sphere
pns_S2_1 = pns_clusterer.fit_pns_to_spherical_data(theta1, phi1)
theta1_new, phi1_new = pns_S2_1.dists_

# Transform second sphere
pns_S2_2 = pns_clusterer.fit_pns_to_spherical_data(theta2, phi2)
theta2_new, phi2_new = pns_S2_2.dists_

# Transform distance coordinates
theta_d, phi_d = pns_clusterer.transform_distance_coordinates(d2_s, d3_s)

# Create angle matrix
angle_matrix = np.column_stack([
    theta_d + 180,
    phi_d + 180,
    alpha_s,
    theta1_new + 180,
    phi1_new + 180,
    theta2_new + 180,
    phi2_new + 180
])

print(f"Angle matrix shape: {angle_matrix.shape}")

# Refine clusters with PNS
mode_clusters, refinement_metadata = refine_clusters_with_pns(
    scale=config.pns_scale,
    data=angle_matrix,
    cluster_list=clusters,
    outlier_list=outliers,
    min_cluster_size=config.min_cluster_size
)

mode_cluster_sizes = [len(c) for c in mode_clusters]
print(f"Final mode clusters: {len(mode_clusters)} clusters with sizes {mode_cluster_sizes}")

### Visualize Clusters

In [ ]:
# Create cluster visualization
if mode_clusters:
    # Prepare data for plotting
    from mintage_low_res.preprocessing.pucker_analysis import sort_data_into_cluster
    
    sorted_clusters = sorted(mode_clusters, key=len, reverse=True)
    data_by_cluster, cluster_len_list = sort_data_into_cluster(
        scaled_coords_subset, sorted_clusters, config.min_cluster_size
    )
    
    # Create scatter plots
    create_scatter_plots(
        data_by_cluster=data_by_cluster,
        filename=f"{OUTPUT_DIR}/detailed_{pucker_type}_clusters",
        set_title=f"Detailed Analysis - {pucker_type.upper()} Clusters",
        number_of_elements=cluster_len_list,
        legend=True,
        s=30
    )
    
    print(f"Cluster plot saved: {OUTPUT_DIR}/detailed_{pucker_type}_clusters.png")
    
    # Display the plot
    if os.path.exists(f"{OUTPUT_DIR}/detailed_{pucker_type}_clusters.png"):
        display(Image(f"{OUTPUT_DIR}/detailed_{pucker_type}_clusters.png"))
else:
    print("No clusters found to visualize")

## Method 3: Q-Fold Parameter Optimization

Run q-fold optimization experiments as in the original notebook.

In [ ]:
# Run q-fold optimization experiments
print("Running q-fold optimization experiments...")

q_fold_results = run_pucker_analysis(
    suites=suites,
    scaled_coords=scaled_coords,
    pucker_types=['c2c2', 'c3c3'],  # Run for subset to save time
    output_dir=f"{OUTPUT_DIR}/pucker_analysis",
    find_optimal_q_fold=True
)

print("\nQ-fold optimization complete!")
print("\nPucker Statistics:")
for pucker_type, stats in q_fold_results['pucker_statistics'].items():
    print(f"  {pucker_type}: {stats['n_suites']} suites ({stats['percentage']:.1f}%)")

if 'q_fold_summary' in q_fold_results:
    print("\nOptimal Q-fold Values:")
    for pucker_type, q_fold in q_fold_results['q_fold_summary']['optimal_q_folds'].items():
        print(f"  {pucker_type}: {q_fold}")

## Summary and Comparison

Compare the new organized approach with the original notebook approach.

In [ ]:
print("=== ANALYSIS SUMMARY ===")
print("\nBenefits of the new organized approach:")
print("✓ Clean, modular code structure")
print("✓ Automatic caching of expensive computations")
print("✓ Comprehensive error handling and logging")
print("✓ Type hints and documentation")
print("✓ Configurable parameters")
print("✓ Reproducible experiments")
print("✓ Easy to extend and modify")

print("\nOriginal notebook functionality preserved:")
print("✓ Same PDB parsing and Suite creation")
print("✓ Same coordinate scaling approach")
print("✓ Same hierarchical clustering algorithm")
print("✓ Same PNS transformation and refinement")
print("✓ Same visualization and plotting")
print("✓ Same q-fold optimization experiments")

if 'results' in locals():
    print(f"\nExecution completed successfully in {results['total_time']:.1f} seconds")
    print(f"Results saved to: {OUTPUT_DIR}/")
    print(f"Cache saved to: {CACHE_DIR}/")

## Next Steps

This clean notebook demonstrates how to use the refactored mintage_low_res package. You can now:

1. **Modify parameters** easily through the `ExperimentConfig` class
2. **Add new pucker types** or analysis methods by extending the appropriate modules
3. **Run experiments** with different datasets by changing the PDB directory
4. **Customize visualizations** by modifying the plotting functions
5. **Add new clustering algorithms** by implementing the clustering interface
6. **Cache intermediate results** to speed up repeated experiments
7. **Run in parallel** or on clusters by using the modular structure

The modular design makes it easy to understand, modify, and extend the analysis pipeline while maintaining reproducibility and performance.